# Training the Models

This notebook showcases the training process of the two model variants.


In [1]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), "src"))
data_path = os.path.join(os.getcwd(), "dataset")

import numpy as np
from datasets import load_dataset

from model import SkipGramW2V, CBOWW2V
from dataloader import DataLoader
from train import train_skipgram_ns, train_cbow_ns

In [2]:
# Load the WikiText-2 dataset
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", cache_dir=data_path, trust_remote_code=True)
train_dataset = dataset["train"]
del dataset

In [3]:
def preprocess_text(text):
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = text.replace(".", " . ").replace(",", " , ").replace("!", " ! ").replace("?", " ? ")
    text = text.lower()
    return text

# WikiText has multiple rows, each with a 'text' field — keep non-empty lines
train_texts = [preprocess_text(row["text"]) for row in train_dataset if row["text"].strip()]

# Create entire train text by concatenating all lines, to build the vocabulary.
train_text = " ".join(train_texts)
dataloader = DataLoader(train_text, vocab_size=10000, context_window=4)
del train_text

In [4]:
skipgram = SkipGramW2V(vocab_size=dataloader.vocab_size, embedding_dim=128)
cbow = CBOWW2V(vocab_size=dataloader.vocab_size, embedding_dim=128)

In [5]:
train_skipgram_ns(skipgram, dataloader, train_texts, epochs=10, learning_rate=0.01, k=10, print_every=1)

Epoch 0/10, Loss: 3.1054
Epoch 1/10, Loss: 2.4269
Epoch 2/10, Loss: 2.4011
Epoch 3/10, Loss: 2.3876
Epoch 4/10, Loss: 2.3751
Epoch 5/10, Loss: 2.3617
Epoch 6/10, Loss: 2.3471
Epoch 7/10, Loss: 2.3298
Epoch 8/10, Loss: 2.3097
Epoch 9/10, Loss: 2.2904


In [6]:
train_cbow_ns(cbow, dataloader, train_texts, epochs=10, learning_rate=0.01, k=10, print_every=1)

Epoch 0/10, Loss: 3.8652
Epoch 1/10, Loss: 3.0801
Epoch 2/10, Loss: 2.7752
Epoch 3/10, Loss: 2.6382
Epoch 4/10, Loss: 2.5745
Epoch 5/10, Loss: 2.5378
Epoch 6/10, Loss: 2.5146
Epoch 7/10, Loss: 2.4993
Epoch 8/10, Loss: 2.4835
Epoch 9/10, Loss: 2.4688


In [7]:
cbow.save_model("models/cbow_model.npz")
skipgram.save_model("models/skipgram_model.npz")